In [1]:
import time
import httpx
import numpy as np
from wearable_affect.api import WristSignals
from wearable_affect.data import DEFAULT_DATA_DIR, PROJECT_ROOT, WRIST_FS, load_subject
from wearable_affect.detector import StressDetector
from wearable_affect.windows import make_windows

API = "http://127.0.0.1:8000"
rec = load_subject(DEFAULT_DATA_DIR, "S2")
windows = make_windows(rec)
start = windows[0].start_s
calibration = {name: rec.signals[name][start * fs : (start + 300) * fs] for name, fs in WRIST_FS.items()}

response = httpx.post(f"{API}/calibrate", json=WristSignals.from_arrays(calibration).model_dump(), timeout=30)
print(response.status_code, response.json()["n_windows"])
baseline = response.json()["baseline"]

200 9


In [2]:
detector = StressDetector.load(PROJECT_ROOT / "artifacts" / "stress_detector.joblib")
direct_baseline = detector.baseline_from_calibration(calibration)

latencies, differences = [], []
for w in windows:
    body = {"signals": WristSignals.from_arrays(w.signals).model_dump(), "baseline": baseline}
    t0 = time.perf_counter()
    r = httpx.post(f"{API}/predict", json=body, timeout=30)
    latencies.append(time.perf_counter() - t0)
    differences.append(abs(r.json()["stress_probability"] - detector.predict_proba(w.signals, direct_baseline)))

print(f"largest difference, API vs direct: {max(differences):.1e}")
print(f"latency: median {np.median(latencies) * 1000:.0f} ms, 95th percentile {np.percentile(latencies, 95) * 1000:.0f} ms")

largest difference, API vs direct: 2.4e-06
latency: median 408 ms, 95th percentile 660 ms
